In [1]:
# 05_explain_lime.ipynb
# Post-hoc Explainability using LIME

import sys
import time
import psutil
import tracemalloc
import os
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


from lime.lime_tabular import LimeTabularExplainer
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
lime_results_path = Path("../results/LIME")
lime_results_path.mkdir(parents=True, exist_ok=True)

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=X.columns.tolist(),
        class_names=["No Defect", "Defect"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_STATE
    )

    defect_idx = np.where(y_test.values == 1)[0]
    if len(defect_idx) == 0:
        print(f"  No defective instances in {ds}")
        continue
    selected_idx = defect_idx[:N_INSTANCES]

    dataset_model_path = models_path / ds
    if not dataset_model_path.exists():
        continue

    for model_dir in sorted(dataset_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f" → {model_name}")

        model = joblib.load(model_dir / "model.joblib")

        save_path = lime_results_path / ds / model_name
        save_path.mkdir(parents=True, exist_ok=True)

        records = []
        lime_times = []

        _mem_before_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        tracemalloc.start()

        for i, idx in enumerate(selected_idx):
            instance = X_test.iloc[idx].values

            _t0 = time.perf_counter()
            explanation = explainer.explain_instance(
                data_row=instance,
                predict_fn=model.predict_proba
            )

            lime_times.append(time.perf_counter() - _t0)

            html_file = save_path / f"lime_instance_{i}.html"
            explanation.save_to_file(str(html_file))

        _, _peak_traced = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        _mem_after_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        _peak_mb = max(_mem_after_loop - _mem_before_loop, _peak_traced / (1024 * 1024))

        timing_entry = {
            "Dataset": ds, "Model": model_name, "Method": "LIME",
            "Explain_Time_Mean_s": round(float(sum(lime_times)/len(lime_times)), 4),
            "Explain_Time_Std_s":  round(float(pd.Series(lime_times).std()), 4),
            "Mem_Delta_MB": round(_peak_mb, 2)
        }
        timing_csv = save_path / "lime_timing.csv"
        pd.DataFrame([timing_entry]).to_csv(timing_csv, index=False)
        print(f"   LIME time: {timing_entry['Explain_Time_Mean_s']:.4f}s ± {timing_entry['Explain_Time_Std_s']:.4f}s | Peak Mem: {timing_entry['Mem_Delta_MB']:.1f} MB")






DATASET: CM1
 → Gradient_Boosting


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have vali

   LIME time: 0.2461s ± 0.0817s | Peak Mem: 10.6 MB
 → MLP


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted w

   LIME time: 0.2213s ± 0.0148s | Peak Mem: 8.0 MB
 → Random_Forest


...

   LIME time: 0.1623s ± 0.0318s | Peak Mem: 8.0 MB
 → Random_Forest


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names,

   LIME time: 0.4162s ± 0.1181s | Peak Mem: 6.8 MB
 → SVM


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted w

   LIME time: 1.5267s ± 0.5607s | Peak Mem: 6.8 MB
